In [1]:
from src.data import *
from src.features import *
from src.debug import print_dataset_info
from src.models import *
from src.config import MASK_VALUE


import tensorflow as tf
from sklearn import feature_extraction, model_selection
from sklearn.metrics import mean_squared_error, roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.preprocessing import MinMaxScaler

In [2]:
dfs = load_all_data()

In [3]:
print_dataset_info(dfs)


Dataset: documents
Shape: (46771, 10)


,version,document_id,title,type,created_time,author_id,content,status,version_comment,topic_id
0,128,0Ihkl0CyzoWSjDm5F,NaN,MULTIPLE_CHOICE,2018-07-16 12:26:29.402,NaN,"{""id"": ""0Ihkl0CyzoWSjDm5F"", ""type"": ""MULTIPLE_...",WORK_IN_PROGRESS,NaN,3115
1,130,0Ihkl0CyzoWSjDm5F,Aufgabe PT 2.12,MULTIPLE_CHOICE,2018-07-16 12:29:22.905,NaN,"{""id"": ""0Ihkl0CyzoWSjDm5F"", ""type"": ""MULTIPLE_...",WORK_IN_PROGRESS,NaN,3115
2,664,0Ihkl0CyzoWSjDm5F,Aufgabe PT 2.12,MULTIPLE_CHOICE,2018-07-20 09:43:12.855,NaN,"{""id"": ""0Ihkl0CyzoWSjDm5F"", ""type"": ""MULTIPLE_...",WORK_IN_PROGRESS,NaN,3115
3,1161,0Ihkl0CyzoWSjDm5F,Aufgabe PT 2.12,MULTIPLE_CHOICE,2018-07-26 09:29:18.045,NaN,"{""id"": ""0Ihkl0CyzoWSjDm5F"", ""type"": ""MULTIPLE_...",WORK_IN_PROGRESS,NaN,3115
4,1735,0Ihkl0CyzoWSjDm5F,Aufgabe PT 2.12,MULTIPLE_CHOICE,2018-08-22 13:02:14.350,NaN,"{""id"": ""0Ihkl0CyzoWSjDm5F"", ""type"": ""MULTIPLE_...",WORK_IN_PROGRESS,NaN,3115




Dataset: topic_trees
Shape: (678, 5)


,topic_id,parent_id,child_id,sibling_rank,displayed_on_dashboard
0,1059,NaN,1,1,1
1,1060,NaN,109,109,1
2,1061,1.0,2,0,1
3,1062,1.0,2022,11,1
4,1063,3069.0,2559,409,0




Dataset: topics_translated
Shape: (700, 6)


,id,german_name,german_description,name,description,math
0,1,Deutsch,Sprache als System,German,Language as a system,0
1,2,Orthografie,NaN,Orthography,NaN,0
2,3,Rechtschreibprinzipien,NaN,Spelling principles,NaN,0
3,109,Mathematik,Rechnen und so...,Mathematics,Calculating and such...,1
4,950,Zahlen und Zahlenmengen,NaN,Numbers and number sets,NaN,1




Dataset: transactions
Shape: (2134759, 21)


,transaction_id,transaction_token,user_id,document_id,document_version,evaluation,input,start_time,commit_time,user_agent,...,type,session_id,topic_id,session_closed,session_type,session_accepted,challenge,challenge_id,challenge_order,challenge_name
0,688413,88fdcaad-f73b-46a2-b561-d262f2441442,393211,awd0i1DlVtg6kuMZSkpmHa,75002,PARTIAL,"{""type"": ""MULTI_COLOR_HIGHLIGHT"", ""highlighted...",2021-05-21 07:58:27.312000000,2021-05-21 08:03:43.020000000,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,MULTI_COLOR_HIGHLIGHT,NaN,NaN,NaN,NaN,NaN,True,1083.0,2.0,G3h – Training Rhetorik
1,688414,a75eb7b4-b2c2-47d4-9200-27980c175037,393211,arhWF3BT53V9W8cGOaZVPX,75012,PARTIAL,"{""type"": ""MULTI_COLOR_HIGHLIGHT"", ""highlighted...",2021-05-21 08:04:05.067000000,2021-05-21 08:07:21.288999936,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,MULTI_COLOR_HIGHLIGHT,NaN,NaN,NaN,NaN,NaN,True,1083.0,3.0,G3h – Training Rhetorik
2,688415,61eb829d-bdda-4107-86af-ad9a14a7bdc9,393211,9wk5dtV2mF59odW0wCEYYc,75003,PARTIAL,"{""type"": ""CLOZE_TEXT"", ""clozeInputs"": [""Person...",2021-05-21 08:07:37.048000000,2021-05-21 08:13:30.953999872,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,CLOZE_TEXT,NaN,NaN,NaN,NaN,NaN,True,1083.0,4.0,G3h – Training Rhetorik
3,688416,30ff0d8a-865d-460b-9177-b698a52b0d5c,393211,afilxZ8LycP5LReULeKngW,75009,CORRECT,"{""type"": ""DND_PAIRS"", ""input"": [""<p>Ich gehe i...",2021-05-21 08:13:38.943000000,2021-05-21 08:22:13.975000064,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,DND_PAIRS,NaN,NaN,NaN,NaN,NaN,True,1083.0,5.0,G3h – Training Rhetorik
4,688417,0adedf3b-ba35-4497-8c6b-b5c2f6fcbbf3,393211,76m6v05NCeX8x2Wr5tKRE3,75007,CORRECT,"{""type"": ""DND_PAIRS"", ""input"": [""<p>Kleiner Ma...",2021-05-21 08:22:19.391000000,2021-05-21 08:22:55.366000128,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,DND_PAIRS,NaN,NaN,NaN,NaN,NaN,True,1083.0,6.0,G3h – Training Rhetorik




Dataset: users
Shape: (30929, 6)


,user_id,gender,canton,class_level,study,class_id
0,387604,NaN,NaN,NaN,False,NaN
1,387605,NaN,NaN,NaN,False,NaN
2,387608,NaN,NaN,NaN,True,9Q2M7
3,387613,NaN,NaN,NaN,False,NaN
4,387615,MALE,SG,Gymnasium - 3. Jahr,False,NaN


In [4]:
documents = get_latest_documents(documents=dfs['documents'])
display(documents.head())

,version,document_id,title,type,created_time,content,topic_id,estimatedDuration,estimatedDifficulty
2262,487,fTp7AX3QQfx8vvTLNKGPRv,Aufgabe PT 7.12f,CLOZE_TEXT_DROPDOWN,2018-07-18 14:30:57.539,"{""id"": ""fTp7AX3QQfx8vvTLNKGPRv"", ""type"": ""CLOZ...",1,NaN,NaN
2250,590,fqjEP599AZ29AtN0XNgWxT,Aufgabe PT 5.6,CLUSTER,2018-07-19 13:38:58.442,"{""id"": ""fqjEP599AZ29AtN0XNgWxT"", ""type"": ""CLUS...",1,NaN,NaN
2151,676,etfOjQpk4os8hF9z0rMfwt,Aufgabe 5.10a,MULTIPLE_CHOICE,2018-07-20 10:40:15.845,"{""id"": ""etfOjQpk4os8hF9z0rMfwt"", ""type"": ""MULT...",1,NaN,NaN
1612,677,aVwP8ywckxMbAH-8w2wfMe,Aufgabe PT 5.10d,MULTIPLE_CHOICE,2018-07-20 10:41:00.154,"{""id"": ""aVwP8ywckxMbAH-8w2wfMe"", ""type"": ""MULT...",1,NaN,NaN
1742,680,cEY-K2TJ4HGaNRqp5bjhF5,Aufgabe PT 7.11b,CLOZE_TEXT_DROPDOWN,2018-07-20 11:01:29.324,"{""id"": ""cEY-K2TJ4HGaNRqp5bjhF5"", ""type"": ""CLOZ...",1,NaN,NaN


In [5]:
parsed = dfs['documents']['content'].apply(lambda c: json.loads(c) if isinstance(c, str) else (c or {}))
parsed[18]['metaData']

{'isMobile': True, 'estimatedDuration': 120, 'estimatedDifficulty': 3}

In [6]:
summarize_documents(documents)

Total distinct documents: 5746
Documents with estimatedDuration and estimatedDifficulty: 4735 (82.4%)
Filtered out: 1011 (17.6%)
Distinct topics: 370


['fTp7AX3QQfx8vvTLNKGPRv',
 'fqjEP599AZ29AtN0XNgWxT',
 'etfOjQpk4os8hF9z0rMfwt',
 'aVwP8ywckxMbAH-8w2wfMe',
 'cEY-K2TJ4HGaNRqp5bjhF5',
 'bKyFkknHAW7aeH4d0qhpEn',
 '1fsDFKDLk8KbZ2xBBzNx--',
 '8JmheE6M4208C3iLPQHRnP',
 '1eUS4GFW4.g8vLxi-sRo1z',
 'bNPQokamA9sbXPqYi8fSzh',
 'awGP3BNZk5UbMHbTW4R5x3',
 '59scgU0wkw-8qdY1ZP59DI',
 'Af0RVdAEibQAr.M9AVTj',
 'ffTij6YYA-zbcx9ZP8QWLy',
 '4HNL62l.47Ta0sHqrOzKoF',
 'aCF0hfb1kQEayDOqwEp0s1',
 '78debrP047o8YoDB0nymp2',
 'dCZ4cWixQrM9ovYwc-TP.z',
 'Bx98geNrvUYWSjsh8g',
 '6f7aujnwQTxb56qwxs1sFX',
 'bVubBjOtA6x9x3pOLGQL95',
 'ctu8cT8ZQVebZn4G3nux8s',
 'dNMQxmWW4pe9auQHzX6DW7',
 '3D4ovRZd4e69keiP-DEyXl',
 '9XwCwbGcQaObYPjBFfQeab',
 'fkcsHMUrkcvadzh3aQdifI',
 '6shnOMCmAMSasjMJay1KdE',
 '448ge.u7Qzyaj4XuEBTIaH',
 '5oWgAX-Fk82bNdaRxoXBgh',
 '6SUe7t6SkV58p70qoJwl3K',
 '9VnHcCno4dF93oGTP1nYyc',
 '483vpC1jQbk8WYeIpt6T-d',
 '56ljt80hkbH8A4vzX.Je5q',
 '3XEDx-kl4Egb92HW9bD0mC',
 '8TxLI5oe4c29OZzvd4o85a',
 'Li7CRHY4gP98s.i4z1-by',
 '5WUsFXhOQwJ9e8Gm81NMto',
 '7CvyCZ

In [7]:
build_topic_lookups(dfs['topics_translated'], dfs['topic_trees'])['child_to_parent']

{2: 1,
 2022: 1,
 2559: 3069,
 2560: 3069,
 2861: 3077,
 950: 109,
 969: 109,
 985: 109,
 1006: 109,
 1030: 109,
 1045: 109,
 1058: 109,
 2854: 3079,
 2855: 3079,
 2882: 3079,
 2884: 3079,
 2913: 3079,
 3: 2,
 2055: 2,
 2064: 2,
 2065: 2,
 2023: 2022,
 2042: 2022,
 2045: 2022,
 2048: 2022,
 951: 950,
 952: 950,
 956: 950,
 957: 950,
 958: 950,
 962: 950,
 963: 950,
 964: 950,
 968: 950,
 970: 969,
 971: 969,
 972: 969,
 973: 969,
 974: 969,
 978: 969,
 984: 969,
 986: 985,
 987: 985,
 992: 985,
 997: 985,
 1001: 985,
 1002: 985,
 1003: 985,
 1004: 993,
 1005: 985,
 1007: 1006,
 1010: 1006,
 1018: 1006,
 1024: 1006,
 1029: 1006,
 1031: 1030,
 1035: 1030,
 1036: 1030,
 1040: 1030,
 1041: 1030,
 1046: 1045,
 1051: 1045,
 1059: 1058,
 1060: 1058,
 1061: 1058,
 1062: 1058,
 1063: 1058,
 1067: 1058,
 2053: 3,
 2054: 3,
 2009: 3,
 2010: 3,
 2070: 2055,
 2062: 2055,
 2063: 2055,
 3019: 3069,
 3021: 3069,
 3022: 3069,
 2067: 2065,
 2068: 4127,
 2069: 2065,
 2031: 2023,
 2035: 2023,
 2043: 2042,

In [8]:
lookups = build_topic_lookups(dfs['topics_translated'], dfs['topic_trees'])
topics = add_topic_depth(dfs['topics_translated'], lookups['child_to_parent'])
display(topics.head())

Depth distribution across all topics:
       n_topics
depth          
0            25
1            72
2           179
3           294
4           110
5            20


,id,german_name,german_description,name,description,math,depth
0,1,Deutsch,Sprache als System,German,Language as a system,0,0
1,2,Orthografie,NaN,Orthography,NaN,0,1
2,3,Rechtschreibprinzipien,NaN,Spelling principles,NaN,0,2
3,109,Mathematik,Rechnen und so...,Mathematics,Calculating and such...,1,0
4,950,Zahlen und Zahlenmengen,NaN,Numbers and number sets,NaN,1,1


In [9]:
display(docs_per_topic(documents).head(20))
display(docs_per_depth(documents, topics))
display(topics_with_docs_per_depth(documents, topics))

,n_documents
topic_id,
1,319
998,71
988,67
1040,65
955,64
2069,63
1032,62
975,56
956,55


,n_documents
depth,
0,388
1,126
2,1606
3,3348
4,145
5,133


,n_topics_with_docs
depth,
0,7
1,20
2,98
3,187
4,38
5,20


In [10]:
active = documents['topic_id'].unique()
display(topics[(topics['depth'] == 0) & (topics['id'].isin(active))])

,id,german_name,german_description,name,description,math,depth
0,1,Deutsch,Sprache als System,German,Language as a system,0,0
3,109,Mathematik,Rechnen und so...,Mathematics,Calculating and such...,1,0
116,2026,Ober-/Unterbegriff,NaN,Top/sub term,NaN,0,0
117,2027,Synonyme/Antonyme,NaN,Synonyms/antonyms,NaN,0,0
118,2028,Mehrdeutigkeit,NaN,Ambiguity,NaN,0,0
119,2029,Verschiedenes,NaN,Miscellaneous,NaN,0,0
477,3425,Zu löschen,"ungültig, wird gelöscht.",To delete,"invalid, will be deleted.",1,0


In [11]:
dfs['topic_trees'] = reparent_topics(dfs['topic_trees'], [2026, 2027, 2028], new_parent_id=1)
lookups = build_topic_lookups(dfs['topics_translated'], dfs['topic_trees'])
topics = add_topic_depth(dfs['topics_translated'], lookups['child_to_parent'])
display(dfs['topic_trees'][dfs['topic_trees']['child_id'].isin([2026, 2027, 2028])])

Depth distribution across all topics:
       n_topics
depth          
0            22
1            50
2           204
3           294
4           110
5            20


,topic_id,parent_id,child_id,sibling_rank,displayed_on_dashboard
678,6108,1.0,2026,0,0
679,6109,1.0,2027,0,0
680,6110,1.0,2028,0,0


In [12]:
before_tt = len(dfs['topic_trees'])
before_tr = len(dfs['topics_translated'])
dfs['topic_trees'] = prune_empty_topics(dfs['topic_trees'], documents)
dfs['topics_translated'] = prune_empty_topics_translated(dfs['topics_translated'], dfs['topic_trees'], documents)
print(f"topic_trees rows: {before_tt} -> {len(dfs['topic_trees'])}")
print(f"topics_translated rows: {before_tr} -> {len(dfs['topics_translated'])}")
lookups = build_topic_lookups(dfs['topics_translated'], dfs['topic_trees'])
topics = add_topic_depth(dfs['topics_translated'], lookups['child_to_parent'])
topics[topics.depth == 0]

topic_trees rows: 681 -> 378
topics_translated rows: 700 -> 379
Depth distribution across all topics:
       n_topics
depth          
0             4
1            25
2           103
3           188
4            39
5            20


,id,german_name,german_description,name,description,math,depth
0,1,Deutsch,Sprache als System,German,Language as a system,0,0
3,109,Mathematik,Rechnen und so...,Mathematics,Calculating and such...,1,0
116,2029,Verschiedenes,NaN,Miscellaneous,NaN,0,0
297,3425,Zu löschen,"ungültig, wird gelöscht.",To delete,"invalid, will be deleted.",1,0


In [13]:
before_tt = len(dfs['topic_trees'])
before_tr = len(dfs['topics_translated'])
dfs['topic_trees'], dfs['topics_translated'] = drop_topic_subtrees(
    dfs['topic_trees'], dfs['topics_translated'], [2029, 3425]
)
print(f"topic_trees rows: {before_tt} -> {len(dfs['topic_trees'])}")
print(f"topics_translated rows: {before_tr} -> {len(dfs['topics_translated'])}")
lookups = build_topic_lookups(dfs['topics_translated'], dfs['topic_trees'])
topics = add_topic_depth(dfs['topics_translated'], lookups['child_to_parent'])

topic_trees rows: 378 -> 377
topics_translated rows: 379 -> 377
Depth distribution across all topics:
       n_topics
depth          
0             2
1            25
2           103
3           188
4            39
5            20


# Transaction preprocessing

In [14]:
evaluated = summarize_transactions(dfs['transactions'], topics, documents)
display(evaluated.head())

Total transactions: 2134759
Evaluated transactions: 1401007 (65.6%)
Evaluated with unknown topic_id: 348782 (24.9%)
Evaluated with no estimatedDifficulty: 34228 (2.4%)


,transaction_id,transaction_token,user_id,document_id,document_version,evaluation,input,start_time,commit_time,user_agent,...,session_id,topic_id,session_closed,session_type,session_accepted,challenge,challenge_id,challenge_order,challenge_name,estimatedDifficulty
0,688413,88fdcaad-f73b-46a2-b561-d262f2441442,393211,awd0i1DlVtg6kuMZSkpmHa,75002,PARTIAL,"{""type"": ""MULTI_COLOR_HIGHLIGHT"", ""highlighted...",2021-05-21 07:58:27.312000000,2021-05-21 08:03:43.020000000,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,2.0,G3h – Training Rhetorik,3.0
1,688414,a75eb7b4-b2c2-47d4-9200-27980c175037,393211,arhWF3BT53V9W8cGOaZVPX,75012,PARTIAL,"{""type"": ""MULTI_COLOR_HIGHLIGHT"", ""highlighted...",2021-05-21 08:04:05.067000000,2021-05-21 08:07:21.288999936,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,3.0,G3h – Training Rhetorik,4.0
2,688415,61eb829d-bdda-4107-86af-ad9a14a7bdc9,393211,9wk5dtV2mF59odW0wCEYYc,75003,PARTIAL,"{""type"": ""CLOZE_TEXT"", ""clozeInputs"": [""Person...",2021-05-21 08:07:37.048000000,2021-05-21 08:13:30.953999872,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,4.0,G3h – Training Rhetorik,3.0
3,688416,30ff0d8a-865d-460b-9177-b698a52b0d5c,393211,afilxZ8LycP5LReULeKngW,75009,CORRECT,"{""type"": ""DND_PAIRS"", ""input"": [""<p>Ich gehe i...",2021-05-21 08:13:38.943000000,2021-05-21 08:22:13.975000064,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,5.0,G3h – Training Rhetorik,3.0
4,688417,0adedf3b-ba35-4497-8c6b-b5c2f6fcbbf3,393211,76m6v05NCeX8x2Wr5tKRE3,75007,CORRECT,"{""type"": ""DND_PAIRS"", ""input"": [""<p>Kleiner Ma...",2021-05-21 08:22:19.391000000,2021-05-21 08:22:55.366000128,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,6.0,G3h – Training Rhetorik,2.0


In [15]:
math_df, german_df = split_by_subject(evaluated, topics)
print(f"math: {len(math_df)} rows | german: {len(german_df)} rows")

# Pipeline-test sample: keep at most 100 distinct users per subset.
math_df = math_df[math_df['user_id'].isin(math_df['user_id'].drop_duplicates().head(100))].reset_index(drop=True)
german_df = german_df[german_df['user_id'].isin(german_df['user_id'].drop_duplicates().head(100))].reset_index(drop=True)
print(f"sampled math: {len(math_df)} rows ({math_df['user_id'].nunique()} users)")
print(f"sampled german: {len(german_df)} rows ({german_df['user_id'].nunique()} users)")

math: 293465 rows | german: 758760 rows
sampled math: 2163 rows (100 users)
sampled german: 3971 rows (100 users)


In [16]:
X = build_feature_matrix(math_df, documents)
display(X)

,user_id,skill,evaluation,start_time,skill_attempts,total_attempts
0,390142,1046_2,CORRECT,2021-05-21 10:16:09.924000000,0,0
1,390137,1059_1,PARTIAL,2021-05-21 10:16:58.803000000,0,0
2,390140,987_1,WRONG,2021-05-21 10:20:53.223000000,0,0
3,390140,987_2,WRONG,2021-05-21 10:23:24.005000000,0,1
4,390140,987_1,WRONG,2021-05-21 10:24:10.729000000,1,2
...,...,...,...,...,...,...
2158,388487,1007_2,CORRECT,2023-01-14 17:40:13.256000000,6,100
2159,388487,1007_2,CORRECT,2023-01-14 17:40:21.597000000,7,101
2160,388487,1007_2,CORRECT,2023-01-14 17:40:28.418000000,8,102
2161,388487,1007_2,CORRECT,2023-01-14 17:40:59.389000000,9,103


In [17]:
print("Number of unique students in the dataset:", len(set(X['user_id'])))
print("Number of unique skills in the dataset:", len(set(X['skill'])))


Number of unique students in the dataset: 100
Number of unique skills in the dataset: 122


In [18]:
params = {}
params['batch_size'] = 8
params['mask_value'] = MASK_VALUE

In [19]:
# Obtain indexes for training and test sets
train_index, test_index = next(create_iterator(X))

# Split the data into training and test
X_train, X_test = X.iloc[train_index], X.iloc[test_index]

# Obtain indexes for training and validation sets
train_val_index, val_index = next(create_iterator(X_train))

# Split the training X into training and validation
X_train_val, X_val = X_train.iloc[train_val_index], X_train.iloc[val_index]

In [20]:
# Build TensorFlow sequence datasets for training, validation, and test data
seq, features_depth, skill_depth = prepare_seq(X)
seq_train = seq[X_train_val.user_id.unique()]
seq_val = seq[X_val.user_id.unique()]
seq_test = seq[X_test.user_id.unique()]

# Prepare the training, validation, and test data in the DKT input format
tf_train, length = prepare_data(seq_train, params, features_depth, skill_depth)
tf_val, val_length  = prepare_data(seq_val, params, features_depth, skill_depth)
tf_test, test_length = prepare_data(seq_test, params, features_depth, skill_depth)

# Calculate the length of each of the train-test-val sets and store as parameters
params['train_size'] = int(length // params['batch_size'])
params['val_size'] = int(val_length // params['batch_size'])
params['test_size'] = int(test_length // params['batch_size'])

2026-04-25 19:55:00.799815: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-04-25 19:55:00.806369: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-04-25 19:55:00.808015: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-04-25 19:55:00.808227: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-25 19:55:00.808905: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [21]:
params['verbose'] = 1

params['best_model_weights'] = 'weights/bestmodel.weights.h5' 

params['optimizer'] = 'adam'

params['recurrent_units'] = 16

params['epochs'] = 1

params['dropout_rate'] = 0.1

In [22]:
params

{'batch_size': 8,
 'mask_value': -1.0,
 'train_size': 1,
 'val_size': 0,
 'test_size': 0,
 'verbose': 1,
 'best_model_weights': 'weights/bestmodel.weights.h5',
 'optimizer': 'adam',
 'recurrent_units': 16,
 'epochs': 1,
 'dropout_rate': 0.1}

In [23]:
dkt_lstm = create_model_lstm(features_depth, skill_depth, params)
dkt_lstm.summary()

print(dkt_lstm.name)

history = train_dkt(dkt_lstm, tf_train, tf_val, params)


/opt/homebrew/Caskroom/miniconda/base/envs/mlbd/lib/python3.12/site-packages/keras/src/layers/layer.py:1035: UserWarning: Layer 'reshape' (of type Reshape) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "DKT"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ features            │ (None, None, 366) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None, 366) │          0 │ features[0][0]    │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking (Masking)   │ (None, None, 366) │          0 │ features[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any (Any)           │ (None, None)      │          0 │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, None, 16)  │     24,512 │ masking[0][0],    │
│                     │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, None, 363) │      6,171 │ lstm[0][0],       │
│ (TimeDistributed)   │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, None, 121, │          0 │ time_distributed… │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ next_skill          │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ outputs             │ (None, None, 3)   │          0 │ reshape[0][0],    │
│ (GatherSkill)       │                   │            │ next_skill[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 30,683 (119.86 KB)

 Trainable params: 30,683 (119.86 KB)

 Non-trainable params: 0 (0.00 B)

DKT


2026-04-25 19:55:02.444463: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-04-25 19:55:03.116441: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: INVALID_ARGUMENT: Incompatible shapes: [8,186] vs. [1,8]
	 [[{{function_node __inference_one_step_on_data_3648}}{{node gradient_tape/DKT_1/outputs_1/add_2}}]]
2026-04-25 19:55:03.116460: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 13822037710286209202
2026-04-25 19:55:03.116465: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: INVALID_ARGUMENT: Incompatible shapes: [8,186] vs. [1,8]
	 [[{{function_node __inference_one_step_on_data_3648}}{{node gradient_tape/DKT_1/outputs_1/add_2}}]]
	 [[Func/StatefulPartitionedCall/gradient_tape/DKT_1/time_distributed_1/loop_body/strided_slice/pfor/while/DKT_1/time_distributed_1

InvalidArgumentError: Graph execution error:

Detected at node gradient_tape/DKT_1/outputs_1/add_2 defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/opt/homebrew/Caskroom/miniconda/base/envs/mlbd/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/opt/homebrew/Caskroom/miniconda/base/envs/mlbd/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/opt/homebrew/Caskroom/miniconda/base/envs/mlbd/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/opt/homebrew/Caskroom/miniconda/base/envs/mlbd/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/opt/homebrew/Caskroom/miniconda/base/envs/mlbd/lib/python3.12/asyncio/base_events.py", line 645, in run_forever

  File "/opt/homebrew/Caskroom/miniconda/base/envs/mlbd/lib/python3.12/asyncio/base_events

In [ ]:
# We load the LSTM model with the best performance, and evaluate it on the test set. 
dkt_lstm.load_weights(params['best_model_weights'])
dkt_lstm.evaluate(tf_test, steps=params['test_size'], verbose=params['verbose'], return_dict=True)

In [ ]:
import tensorflow as tf

# Simulated batch matching real shapes
y_true = tf.random.uniform((8, 186, 122), maxval=2, dtype=tf.int32)
y_true = tf.cast(y_true, tf.float32)
y_pred = tf.random.normal((8, 186, 121, 3))

print("y_true:", y_true.shape)
print("y_pred:", y_pred.shape)

label, logits = get_target(y_true, y_pred, mask_value=MASK_VALUE)
print("label:", label.shape)
print("logits:", logits.shape)


def _valid_mask(y_true):
    """Flag real (non-padded) timesteps by summing the skill one-hot."""
    return tf.reduce_sum(y_true[..., :-1], axis=-1)


mask = _valid_mask(y_true)
print("mask:", mask.shape)

loss = tf.keras.losses.sparse_categorical_crossentropy(label, logits, from_logits=True)
print("loss before masking:", loss.shape)

result = tf.reduce_sum(loss * mask) / tf.reduce_sum(mask)
print("final loss:", result.shape, result.numpy())

In [ ]:
print("Output layer supports_masking:", dkt_lstm.layers[-1].supports_masking)
print("Output layer compute_mask:", dkt_lstm.layers[-1].compute_mask(dkt_lstm.input))

In [ ]:
for x_batch, y_batch in tf_train.take(1):
    y_pred = dkt_lstm(x_batch)
    loss = CustomSparseCategoricalCrossEntropy(y_batch, y_pred)
    print("Forward loss:", loss.numpy())